# GateGuard Assignment-3 — Module A: Transaction Engine Tests

**CS432 Databases | IIT Gandhinagar | 2025-26 Semester II**

This notebook validates the full ACID behaviour of the custom transaction engine built on top of the B+ Tree database.

| Test | Property Verified |
|------|------------------|
| Cell 2 | **Atomicity** — mid-transaction failure causes complete rollback |
| Cell 3 | **Durability** — committed data survives a simulated restart |
| Cell 4 | **Multi-table atomicity** — two tables rollback together |
| Cell 5 | **Consistency** — B+ Tree internal state after mixed operations |

In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from database.db_manager  import DatabaseManager
from database.wal         import WALWriter
from database.transaction import TransactionManager
from database.recovery    import RecoveryManager
from database.consistency import ConsistencyChecker

WAL_PATH = "test_gateguard_wal.log"

# Fresh WAL for each test run
if os.path.exists(WAL_PATH):
    os.remove(WAL_PATH)
    print(f"Removed existing WAL: {WAL_PATH}")

db  = DatabaseManager.create_gateguard_db()
wal = WALWriter(WAL_PATH)
tm  = TransactionManager(db, wal)
cc  = ConsistencyChecker(db)

print("\n✅ Setup complete.")
print(f"   DatabaseManager : {db}")
print(f"   WAL file        : {WAL_PATH}")

In [ ]:
# ── Cell 2: Atomicity Test ─────────────────────────────────────────────────
# Property : Atomicity
# Scenario : Insert a Member, then simulate a crash before inserting PersonVisit.
# Expected : Member must NOT exist after rollback (no partial state).

print("=" * 60)
print("TEST 1 — Atomicity: rollback on simulated mid-transaction failure")
print("=" * 60)

ATOMICITY_MEMBER_ID = 999

try:
    with tm.transaction("atomicity_test") as txn:
        txn.insert("Member", {
            "MemberID": ATOMICITY_MEMBER_ID, "Name": "Test Rollback User",
            "Email": "rollback@iitgn.ac.in", "ContactNumber": "9999999999",
            "Image": None, "Age": 22, "Department": "CS", "TypeID": 1,
            "CreatedAt": "2026-03-26", "UpdatedAt": "2026-03-26"
        })
        print(f"\nAfter INSERT (inside txn): Member {ATOMICITY_MEMBER_ID} = {db.get_table('Member').select(ATOMICITY_MEMBER_ID)}")
        
        # ↓ Simulated crash — imagine network failure / disk error here
        raise RuntimeError("Simulated crash: PersonVisit service unreachable")
        
        # This never runs:
        txn.insert("PersonVisit", {"VisitID": 999, "MemberID": 999})

except RuntimeError as e:
    print(f"\nException caught (expected): {e}")

# Verify
member_after = db.get_table("Member").select(ATOMICITY_MEMBER_ID)
assert member_after is None, f"❌ FAIL: Member {ATOMICITY_MEMBER_ID} still exists: {member_after}"
print(f"\n✅ PASS: Member {ATOMICITY_MEMBER_ID} is absent after rollback — no partial state.")

cc.check_all()

In [ ]:
# ── Cell 3: Durability Test ────────────────────────────────────────────────
# Property : Durability
# Scenario : Commit a real transaction, then simulate a full restart
#            by creating a brand-new DatabaseManager from the WAL.
# Expected : Both Member and PersonVisit are recovered in the new instance.

print("=" * 60)
print("TEST 2 — Durability: committed data survives simulated restart")
print("=" * 60)

DURABILITY_MEMBER_ID = 1
DURABILITY_VISIT_ID  = 1

# Commit a real multi-table transaction
with tm.transaction("durability_test") as txn:
    txn.insert("Member", {
        "MemberID": DURABILITY_MEMBER_ID, "Name": "Dr. Rajesh Kumar",
        "Email": "rajesh@iitgn.ac.in", "ContactNumber": "9876543210",
        "Image": None, "Age": 45, "Department": "CS", "TypeID": 1,
        "CreatedAt": "2026-03-26", "UpdatedAt": "2026-03-26"
    })
    txn.insert("PersonVisit", {
        "VisitID": DURABILITY_VISIT_ID, "MemberID": DURABILITY_MEMBER_ID,
        "EntryGateID": 1, "ExitGateID": None,
        "EntryTime": "2026-03-26T08:00:00", "ExitTime": None, "VehicleID": None
    })

print(f"\nCommitted. Member {DURABILITY_MEMBER_ID} = {db.get_table('Member').select(DURABILITY_MEMBER_ID)['Name']}")
print(f"Committed. Visit  {DURABILITY_VISIT_ID} = {db.get_table('PersonVisit').select(DURABILITY_VISIT_ID)}")

# ── Simulate restart: brand new DatabaseManager (empty memory) ──────────
print("\n── Simulating process restart (fresh empty DatabaseManager)... ──")
db2  = DatabaseManager.create_gateguard_db()   # completely empty
rm   = RecoveryManager(db2, WAL_PATH)
summary = rm.recover()

# Verify Member recovered
rec_member = db2.get_table("Member").select(DURABILITY_MEMBER_ID)
assert rec_member is not None, f"❌ FAIL: Member {DURABILITY_MEMBER_ID} NOT recovered"
assert rec_member["Name"] == "Dr. Rajesh Kumar", f"❌ FAIL: Name mismatch: {rec_member['Name']}"
print(f"✅ PASS: Member {DURABILITY_MEMBER_ID} recovered → Name='{rec_member['Name']}'")

# Verify PersonVisit recovered
rec_visit = db2.get_table("PersonVisit").select(DURABILITY_VISIT_ID)
assert rec_visit is not None, f"❌ FAIL: PersonVisit {DURABILITY_VISIT_ID} NOT recovered"
print(f"✅ PASS: PersonVisit {DURABILITY_VISIT_ID} recovered → VisitID={rec_visit['VisitID']}")

# Consistency check on recovered DB
cc2 = ConsistencyChecker(db2)
result = cc2.check_all()
assert result["overall_ok"], "❌ FAIL: Consistency failed on recovered database"
print("✅ PASS: Recovered database is internally consistent.")

In [ ]:
# ── Cell 4: Multi-Table Atomicity ──────────────────────────────────────────
# Property : Atomicity across multiple tables
# Scenario : Insert into both Member AND PersonVisit, then raise an exception.
# Expected : Both tables roll back — record counts unchanged.

print("=" * 60)
print("TEST 3 — Multi-table atomicity: both tables roll back together")
print("=" * 60)

before_members = db.get_table("Member").count()
before_visits  = db.get_table("PersonVisit").count()
print(f"Before: Member count={before_members}, PersonVisit count={before_visits}")

try:
    with tm.transaction("multi_table_atomic") as txn:
        txn.insert("Member", {
            "MemberID": 200, "Name": "Priya Singh", "Email": "priya@iitgn.ac.in",
            "ContactNumber": "8888888888", "Image": None, "Age": 28,
            "Department": "EE", "TypeID": 1,
            "CreatedAt": "2026-03-26", "UpdatedAt": "2026-03-26"
        })
        txn.insert("PersonVisit", {
            "VisitID": 200, "MemberID": 200, "EntryGateID": 2,
            "ExitGateID": None, "EntryTime": "2026-03-26T09:00:00",
            "ExitTime": None, "VehicleID": None
        })
        raise ValueError("Simulated: audit log service down — rollback both tables")

except ValueError as e:
    print(f"\nCaught (expected): {e}")

after_members = db.get_table("Member").count()
after_visits  = db.get_table("PersonVisit").count()

assert after_members == before_members, \
    f"❌ FAIL: Member count changed: {before_members} → {after_members}"
assert after_visits == before_visits, \
    f"❌ FAIL: PersonVisit count changed: {before_visits} → {after_visits}"

print(f"\n✅ PASS: Member count ({after_members}) unchanged.")
print(f"✅ PASS: PersonVisit count ({after_visits}) unchanged.")

cc.check_all()

In [ ]:
# ── Cell 5: Consistency After Mixed Operations ─────────────────────────────
# Property : Consistency (B+ Tree internal correctness)
# Scenario : Commit several inserts, updates, and a delete. Then run the
#            consistency checker on every table.
# Expected : Zero issues in all tables — leaf scan matches search path.

print("=" * 60)
print("TEST 4 — Consistency: B+ Tree state after insert / update / delete")
print("=" * 60)

# Insert 5 members
with tm.transaction("bulk_insert") as txn:
    for i in range(10, 15):
        txn.insert("Member", {
            "MemberID": i, "Name": f"Member_{i}", "Email": f"m{i}@iitgn.ac.in",
            "ContactNumber": f"9000000{i:03d}", "Image": None,
            "Age": 20 + i, "Department": "CS", "TypeID": 1,
            "CreatedAt": "2026-03-26", "UpdatedAt": "2026-03-26"
        })
print("Inserted Members 10–14.")

# Update 2 of them
with tm.transaction("update_ops") as txn:
    txn.update("Member", 10, {"Department": "Mech", "Age": 31})
    txn.update("Member", 11, {"Email": "updated_m11@iitgn.ac.in"})
print("Updated Members 10 and 11.")

# Delete one
with tm.transaction("delete_op") as txn:
    txn.delete("Member", 14)
print("Deleted Member 14.")

# Verify update was applied
m10 = db.get_table("Member").select(10)
assert m10["Department"] == "Mech", f"❌ UPDATE failed for Member 10: {m10}"
print(f"\n✅ Update verified: Member 10 Department = '{m10['Department']}'")

# Verify delete was applied
m14 = db.get_table("Member").select(14)
assert m14 is None, f"❌ DELETE failed: Member 14 still exists: {m14}"
print("✅ Delete verified: Member 14 is absent.")

# Full consistency check
result = cc.check_all()
assert result["overall_ok"], "❌ FAIL: Consistency check detected issues"
print("\n✅ PASS: All tables are internally consistent after mixed operations.")

In [ ]:
# ── Cell 6: WAL Inspection ─────────────────────────────────────────────────
# Print the WAL contents to show the evaluator what the log looks like.

import json
from database.wal import WALReader

print("=" * 60)
print("WAL CONTENTS (showing last 15 records)")
print("=" * 60)

reader  = WALReader(WAL_PATH)
records = reader.read_all()
committed = reader.committed_txn_ids()
uncommitted = reader.uncommitted_txn_ids()

print(f"Total records   : {len(records)}")
print(f"Committed txns  : {len(committed)}")
print(f"Uncommitted txns: {len(uncommitted)} (aborted or rolled back)")
print()

for rec in records[-15:]:
    typ = rec.get('type')
    tid = rec.get('txn_id', '')[-12:]
    seq = rec.get('seq')
    tbl = rec.get('table', '')
    key = rec.get('key', '')
    ts  = rec.get('ts', '')[:19] if rec.get('ts') else ''
    
    if typ in ('BEGIN', 'COMMIT', 'ABORT'):
        print(f"  [{seq:>3}] {typ:<7} txn=...{tid}  {ts}")
    else:
        print(f"  [{seq:>3}] {typ:<7} txn=...{tid}  table={tbl:<15} key={key}")

print(f"\n(WAL file: {WAL_PATH})")